# Experimentation Workspace
Fast sandbox for testing new ideas (losses, mining strategies, etc.) on **real embeddings** from a trained checkpoint.  
Test in seconds, train only when you see good signs.

## Section 0: Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

# Register OmegaConf resolvers (same as trainer.py)
def resolve_tuple(*args): return tuple(args)
try:
    OmegaConf.register_new_resolver("tuple", resolve_tuple)
    OmegaConf.register_new_resolver("eval", eval)
except: pass

# Project imports
from lightning_models import LitTBPS
from lightning_data import TBPSDataModule
from data.bases import ImageDataset, TextDataset
from utils.metrics import rank
from model.objectives import (
    compute_cross_modal_circle,
    compute_constrative,
    compute_citc,
    compute_simclr,
)

print("All imports OK")


In [ ]:
# ============================================================
# USER CONFIG - Edit these paths before running
# ============================================================
CONFIG_PATH = "outputs/<date>/<time>/.hydra/config.yaml"  # Path to your Hydra config
CKPT_PATH   = "checkpoints/your_checkpoint.ckpt"          # Path to your trained checkpoint

# Optional: second checkpoint for A/B comparison (Section 7)
CKPT_PATH_B = None  # e.g. "checkpoints/baseline.ckpt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64       # For feature extraction
NUM_WORKERS = 4

In [ ]:
# Publication-quality plot style
def set_publication_style():
    plt.rcParams.update({
        'font.size': 14, 'axes.labelsize': 16, 'axes.titlesize': 18,
        'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 11,
        'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
        'mathtext.fontset': 'stix', 'lines.linewidth': 2,
    })

set_publication_style()

---
## Section 1: Load Model & Extract Embeddings
Load a trained checkpoint and extract embeddings from the test set into memory.  
All subsequent sections operate on `W` (workspace data dict) — no re-extraction needed.

In [ ]:
def load_model_and_data(config_path, ckpt_path, device=DEVICE):
    """Load config, data module, model, and checkpoint weights."""
    config = OmegaConf.load(config_path)
    OmegaConf.set_struct(config, False)

    dm = TBPSDataModule(config)
    # Full setup (no stage arg) so dm.train_set exists — needed because NACIR
    # (if enabled in the loaded checkpoint) sizes its per-sample buffers to
    # len(dm.train_set). For NACIR-off checkpoints this is harmless overhead.
    dm.setup()

    # Match the current LitTBPS signature used by trainer.py.
    # num_iters_per_epoch is only relevant for the LR scheduler, which this
    # notebook does not use — 100 is a placeholder.
    model = LitTBPS(
        config,
        num_iters_per_epoch=100,
        num_train_samples=len(dm.train_set),
    )
    if config.get("lora", None):
        print("Setting up LoRA...")
        model.setup_lora(config.lora)

    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint['state_dict'].items()}
    # strict=False tolerates both directions: (a) checkpoint from a NACIR run
    # loaded into a NACIR-off model (extra noise_state buffers ignored), and
    # (b) pre-NACIR checkpoint loaded into a NACIR-on model (missing buffers
    # stay at their init values).
    model.load_state_dict(state_dict, strict=False)
    model.to(device).eval()
    print(f"Model loaded from {ckpt_path}")
    return config, dm, model


config, dm, model = load_model_and_data(CONFIG_PATH, CKPT_PATH)


In [ ]:
def extract_embeddings(model, dm, config, device=DEVICE, batch_size=BATCH_SIZE):
    """Extract normalized image & text embeddings from the test set."""
    test_img_loader = DataLoader(
        ImageDataset(dm.dataset.test, is_train=False,
                     image_size=config.aug.img.size, mean=config.aug.img.mean, std=config.aug.img.std),
        batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
    )
    test_txt_loader = DataLoader(
        TextDataset(dm.dataset.test, tokenizer=dm.tokenizer, is_train=False),
        batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
    )

    image_feats, image_pids = [], []
    print("Extracting image features...")
    with torch.no_grad():
        for batch in tqdm(test_img_loader):
            feats = model.get_image_features(batch['images'].to(device))
            image_feats.append(F.normalize(feats, p=2, dim=1).cpu())
            image_pids.append(batch['pids'])
    image_feats = torch.cat(image_feats)
    image_pids = torch.cat(image_pids)

    text_feats, text_pids = [], []
    print("Extracting text features...")
    with torch.no_grad():
        for batch in tqdm(test_txt_loader):
            inputs = {}
            if 'caption_input_ids' in batch:
                inputs['input_ids'] = batch['caption_input_ids'].to(device)
                inputs['attention_mask'] = batch['caption_attention_mask'].to(device)
            else:
                inputs['input_ids'] = batch['input_ids'].to(device)
                inputs['attention_mask'] = batch['attention_mask'].to(device)
            feats = model.get_text_features(inputs)
            text_feats.append(F.normalize(feats, p=2, dim=1).cpu())
            text_pids.append(batch['pids'])
    text_feats = torch.cat(text_feats)
    text_pids = torch.cat(text_pids)

    # Get logit scale and bias from backbone
    logit_scale = model.model.backbone.logit_scale.exp().item()
    logit_bias = model.model.backbone.logit_bias.item() if hasattr(model.model.backbone.logit_bias, 'item') else float(model.model.backbone.logit_bias)

    return {
        "image_feats": image_feats, "text_feats": text_feats,
        "image_pids": image_pids, "text_pids": text_pids,
        "logit_scale": logit_scale, "logit_bias": logit_bias,
    }


W = extract_embeddings(model, dm, config)

In [ ]:
# Sanity check
print(f"Image feats : {W['image_feats'].shape}")
print(f"Text feats  : {W['text_feats'].shape}")
print(f"Unique PIDs : {W['image_pids'].unique().numel()} (img), {W['text_pids'].unique().numel()} (txt)")
print(f"Logit scale : {W['logit_scale']:.4f}")
print(f"Logit bias  : {W['logit_bias']:.4f}")

# Quick R@1 sanity
sim = W['text_feats'] @ W['image_feats'].t()
cmc, mAP, mINP, _ = rank(sim, W['text_pids'], W['image_pids'], max_rank=10, get_mAP=True)
print(f"\nSanity check -- R@1: {cmc[0]:.2f}  R@5: {cmc[4]:.2f}  R@10: {cmc[9]:.2f}  mAP: {mAP:.2f}  mINP: {mINP:.2f}")

---
## Section 2: Similarity Matrix Analysis
Inspect the cross-modal similarity space: statistics, heatmap, and positive/negative distributions.

**Good signs:** Clear separation between positive and negative histograms. Large gap. Low overlap.

In [ ]:
# Compute similarity matrix and masks
sim_matrix = W['text_feats'] @ W['image_feats'].t()
pos_mask = torch.eq(W['text_pids'].view(-1, 1), W['image_pids'].view(1, -1))
neg_mask = ~pos_mask

s_p = sim_matrix[pos_mask].numpy()
s_n = sim_matrix[neg_mask].numpy()

# Statistics table
gap = s_p.mean() - s_n.mean()
overlap = (s_n > s_p.min()).sum() / len(s_n) * 100

print(f"{'':>15} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Median':>8}")
print(f"{'Positive':>15} {s_p.mean():8.4f} {s_p.std():8.4f} {s_p.min():8.4f} {s_p.max():8.4f} {np.median(s_p):8.4f}")
print(f"{'Negative':>15} {s_n.mean():8.4f} {s_n.std():8.4f} {s_n.min():8.4f} {s_n.max():8.4f} {np.median(s_n):8.4f}")
print(f"\nGap (mean_p - mean_n): {gap:.4f}")
print(f"Overlap (% neg > min pos): {overlap:.2f}%")

In [ ]:
# Similarity heatmap (small subset)
N_VIS = 50
fig, ax = plt.subplots(figsize=(8, 7))
sub_sim = sim_matrix[:N_VIS, :N_VIS].numpy()
im = ax.imshow(sub_sim, cmap='viridis', aspect='auto')
# Mark positive pairs
sub_pos = pos_mask[:N_VIS, :N_VIS]
ys, xs = torch.where(sub_pos)
ax.scatter(xs.numpy(), ys.numpy(), c='red', s=8, marker='s', alpha=0.7, label='Positive pair')
ax.set_xlabel('Gallery (Image)')
ax.set_ylabel('Query (Text)')
ax.set_title(f'Similarity Matrix (first {N_VIS} samples)')
ax.legend(loc='upper right')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Positive vs Negative similarity histogram
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(s_n, bins=100, alpha=0.6, color='red', label=f'Negative (n={len(s_n):,})', density=True)
ax.hist(s_p, bins=50, alpha=0.6, color='green', label=f'Positive (n={len(s_p):,})', density=True)
ax.axvline(x=s_p.mean(), color='green', linestyle='--', linewidth=1.5, label=f'mean_p={s_p.mean():.3f}')
ax.axvline(x=s_n.mean(), color='red', linestyle='--', linewidth=1.5, label=f'mean_n={s_n.mean():.3f}')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Density')
ax.set_title('Cross-Modal Similarity Distribution')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 3: Loss Function Playground
Compute all existing losses on the loaded embeddings. Then use the **template cell** to write and test your new loss.

**Good signs:** New loss has a finite value, similar scale to existing losses. Non-zero gradient.

In [ ]:
# Use a random batch-sized subset for loss computation (losses expect batch-scale, not full test set)
LOSS_BATCH = 128
idx_img = torch.randperm(W['image_feats'].shape[0])[:LOSS_BATCH]
idx_txt = torch.randperm(W['text_feats'].shape[0])[:LOSS_BATCH]

# For cross-modal losses, we need matched pairs (same indices, same pids)
# Use text indices and find matching image indices by pid
# Simpler: just use first LOSS_BATCH samples where text and image share indices
B = min(LOSS_BATCH, W['image_feats'].shape[0], W['text_feats'].shape[0])
img_f = W['image_feats'][:B].clone().requires_grad_(True)
txt_f = W['text_feats'][:B].clone().requires_grad_(True)
pids  = W['image_pids'][:B]

logit_scale = W['logit_scale']
logit_bias  = W['logit_bias']

# Build sim_targets for N-ITC (same logic as TBPS.prepare_sim_targets)
sim_targets = torch.eq(pids.view(-1, 1), pids.view(1, -1)).float()
sim_targets_sigmoid = -torch.ones(B, B) + 2 * sim_targets  # -1 or +1
sim_targets_softmax = sim_targets / sim_targets.sum(dim=1, keepdim=True)

print(f"Loss batch size: {B}")
print(f"Unique PIDs in batch: {pids.unique().numel()}")

In [ ]:
# Compute all existing losses on the batch
losses = {}

# Cross-modal Circle Loss (the one actually used in production)
losses['Cross-Modal Circle (m=0.25, g=128)'] = compute_cross_modal_circle(img_f, txt_f, pids, m=0.25, gamma=128).item()
losses['Cross-Modal Circle (m=0.35, g=128)'] = compute_cross_modal_circle(img_f, txt_f, pids, m=0.35, gamma=128).item()

# N-ITC (Sigmoid) — current production variant
losses['N-ITC (Sigmoid)'] = compute_constrative(
    image_features=img_f,
    text_features=txt_f,
    sim_targets=sim_targets_sigmoid,
    logit_scale=logit_scale,
    logit_bias=logit_bias,
    use_sigmoid=True,
).item()

# N-ITC (Softmax) — alternative
losses['N-ITC (Softmax)'] = compute_constrative(
    image_features=img_f,
    text_features=txt_f,
    sim_targets=sim_targets_softmax,
    logit_scale=logit_scale,
    logit_bias=logit_bias,
    use_sigmoid=False,
).item()

# CITC — cyclic image-text contrastive
losses['CITC'] = compute_citc(img_f, txt_f, logit_scale, logit_bias, 0.25, 0.25).item()

# Display
print(f"{'Loss Function':<45} {'Value':>10}")
print("-" * 57)
for name, val in losses.items():
    print(f"{name:<45} {val:10.4f}")


In [ ]:
# Parameter sweep: Circle Loss sensitivity to margin (m) and gamma
margins = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
gammas  = [32, 64, 80, 128, 256]

sweep = np.zeros((len(margins), len(gammas)))
for i, m in enumerate(margins):
    for j, g in enumerate(gammas):
        sweep[i, j] = compute_cross_modal_circle(img_f, txt_f, pids, m=m, gamma=g).item()

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(sweep, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(gammas)))
ax.set_xticklabels(gammas)
ax.set_yticks(range(len(margins)))
ax.set_yticklabels(margins)
ax.set_xlabel(r'$\gamma$ (scale)')
ax.set_ylabel(r'$m$ (margin)')
ax.set_title('Cross-Modal Circle Loss: Parameter Sensitivity')
for i in range(len(margins)):
    for j in range(len(gammas)):
        ax.text(j, i, f'{sweep[i,j]:.1f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Write Your New Loss Here
Edit the cell below. It receives L2-normalized embeddings and PIDs.  
Compare against the baseline table above.

In [ ]:
def my_new_loss(image_features, text_features, pids, **kwargs):
    """
    YOUR NEW LOSS FUNCTION.
    
    Args:
        image_features: (B, 768) L2-normalized image embeddings
        text_features:  (B, 768) L2-normalized text embeddings
        pids:           (B,) integer person IDs
    Returns:
        loss: scalar tensor (must have grad)
    
    Available kwargs: logit_scale, logit_bias, sim_targets, ...
    """
    # Example: replace this with your idea
    sim = image_features @ text_features.t()
    pos_mask = torch.eq(pids.view(-1, 1), pids.view(1, -1)).float()
    neg_mask = 1 - pos_mask
    
    # Placeholder: simple triplet-style margin loss
    s_p = (sim * pos_mask).sum(dim=1) / pos_mask.sum(dim=1).clamp(min=1)
    s_n_max = (sim * neg_mask - 1e9 * (1 - neg_mask)).max(dim=1).values
    margin = 0.3
    loss = F.relu(s_n_max - s_p + margin).mean()
    return loss


# --- Compute and compare ---
_img = img_f.detach().clone().requires_grad_(True)
_txt = txt_f.detach().clone().requires_grad_(True)

new_loss = my_new_loss(_img, _txt, pids, logit_scale=logit_scale, logit_bias=logit_bias)
new_loss.backward()
grad_norm = torch.cat([_img.grad.flatten(), _txt.grad.flatten()]).norm().item()

print(f"Your new loss value : {new_loss.item():.4f}")
print(f"Gradient norm       : {grad_norm:.6f}")
print(f"\n--- Comparison ---")
print(f"Cross-Modal Circle  : {losses.get('Cross-Modal Circle (m=0.35, g=128)', 'N/A')}")
print(f"N-ITC (Sigmoid)     : {losses.get('N-ITC (Sigmoid)', 'N/A')}")

---
## Section 4: Gradient Analysis (Key Diagnostic)
The most important section. Measures whether a loss concentrates gradient energy on **hard negatives**.

**Good signs:** Higher % gradient energy on top-10% hard negatives than N-ITC. Steep gradients in the danger zone.

In [ ]:
def compute_sim_gradients(loss_fn, image_feats, text_feats, pids, **kwargs):
    """
    Compute gradient of loss w.r.t. the similarity matrix.
    Returns: grad_matrix (B, B), loss_value (float)
    """
    img = image_feats.detach().clone()
    txt = text_feats.detach().clone()
    # Compute sim as a leaf tensor
    sim = (img @ txt.t()).requires_grad_(True)
    
    # Build masks
    pid_eq = torch.eq(pids.view(-1, 1), pids.view(1, -1))
    pos_mask = pid_eq.float()
    neg_mask = (~pid_eq).float()
    
    loss = loss_fn(sim, pos_mask, neg_mask, pids, **kwargs)
    loss.backward()
    return sim.grad.detach(), loss.item()


def analyze_hard_neg_gradients(grad_matrix, pos_mask, neg_mask, top_k_pct=0.10):
    """
    Analyze gradient energy distribution on hard vs easy negatives.
    """
    neg_grads = grad_matrix[neg_mask.bool()].abs()
    pos_grads = grad_matrix[pos_mask.bool()].abs()
    
    if neg_grads.numel() == 0:
        return {}
    
    # Hard negatives = top-K% by absolute gradient magnitude
    k = max(1, int(len(neg_grads) * top_k_pct))
    topk_vals, _ = neg_grads.topk(k)
    threshold = topk_vals[-1]
    hard_mask = neg_grads >= threshold
    
    total_energy = (neg_grads ** 2).sum().item()
    hard_energy  = (neg_grads[hard_mask] ** 2).sum().item()
    
    return {
        'hard_neg_energy_pct': hard_energy / (total_energy + 1e-12) * 100,
        'mean_grad_hard_neg': neg_grads[hard_mask].mean().item(),
        'mean_grad_easy_neg': neg_grads[~hard_mask].mean().item(),
        'mean_grad_pos': pos_grads.mean().item(),
        'ratio_hard_vs_easy': neg_grads[hard_mask].mean().item() / (neg_grads[~hard_mask].mean().item() + 1e-12),
    }

print("Gradient analysis utilities ready.")

In [ ]:
# Define loss functions that operate on the similarity matrix directly
def _nitc_sigmoid_from_sim(sim, pos_mask, neg_mask, pids, **kw):
    targets = -torch.ones_like(sim) + 2 * pos_mask  # -1 or +1
    logits = kw.get('logit_scale', 14.0) * sim + kw.get('logit_bias', -10.0)
    return -F.logsigmoid(logits * targets).sum(dim=-1).mean()

def _circle_from_sim(sim, pos_mask, neg_mask, pids, m=0.35, gamma=128, **kw):
    pos_mask_no_diag = pos_mask.clone()
    pos_mask_no_diag.fill_diagonal_(0)
    neg_mask_no_diag = neg_mask.clone()
    neg_mask_no_diag.fill_diagonal_(0)
    s_p = sim[pos_mask_no_diag.bool()]
    s_n = sim[neg_mask_no_diag.bool()]
    if s_p.numel() == 0 or s_n.numel() == 0:
        return torch.tensor(0.0, requires_grad=True)
    alpha_p = torch.clamp_min(-s_p.detach() + 1 + m, min=0.)
    alpha_n = torch.clamp_min(s_n.detach() + m, min=0.)
    logit_p = -gamma * alpha_p * (s_p - (1 - m))
    logit_n =  gamma * alpha_n * (s_n - m)
    return F.softplus(torch.logsumexp(logit_p, dim=0) + torch.logsumexp(logit_n, dim=0))

def _cross_circle_from_sim(sim, pos_mask, neg_mask, pids, m=0.25, gamma=128, **kw):
    s_p = sim[pos_mask.bool()]
    s_n = sim[neg_mask.bool()]
    if s_p.numel() == 0 or s_n.numel() == 0:
        return torch.tensor(0.0, requires_grad=True)
    alpha_p = torch.clamp_min(-s_p.detach() + 1 + m, min=0.)
    alpha_n = torch.clamp_min(s_n.detach() + m, min=0.)
    logit_p = -gamma * alpha_p * (s_p - (1 - m))
    logit_n =  gamma * alpha_n * (s_n - m)
    return F.softplus(torch.logsumexp(logit_p, dim=0) + torch.logsumexp(logit_n, dim=0))

# Template: wrap your new loss to work with sim matrix
def _my_new_loss_from_sim(sim, pos_mask, neg_mask, pids, **kw):
    """Adapt my_new_loss to operate on the similarity matrix."""
    s_p = (sim * pos_mask).sum(dim=1) / pos_mask.sum(dim=1).clamp(min=1)
    s_n_max = (sim * neg_mask - 1e9 * (1 - neg_mask)).max(dim=1).values
    return F.relu(s_n_max - s_p + 0.3).mean()

print("Sim-based loss wrappers ready.")

In [ ]:
# Gradient analysis comparison table
_img_b = img_f.detach()[:B]
_txt_b = txt_f.detach()[:B]
_pids_b = pids[:B]
_pos = torch.eq(_pids_b.view(-1, 1), _pids_b.view(1, -1)).float()
_neg = 1 - _pos

loss_fns = {
    'N-ITC (Sigmoid)': lambda *a, **k: _nitc_sigmoid_from_sim(*a, logit_scale=logit_scale, logit_bias=logit_bias),
    'Circle (m=0.35, g=128)': lambda *a, **k: _circle_from_sim(*a, m=0.35, gamma=128),
    'Cross-Modal Circle': lambda *a, **k: _cross_circle_from_sim(*a, m=0.25, gamma=128),
    'Your New Loss': _my_new_loss_from_sim,
}

print(f"{'Loss':<25} {'Energy% Hard':>14} {'Mean Hard':>10} {'Mean Easy':>10} {'Ratio':>8} {'Mean Pos':>10}")
print("-" * 80)

for name, fn in loss_fns.items():
    grad_mat, loss_val = compute_sim_gradients(fn, _img_b, _txt_b, _pids_b)
    stats = analyze_hard_neg_gradients(grad_mat, _pos, _neg)
    if stats:
        print(f"{name:<25} {stats['hard_neg_energy_pct']:13.1f}% "
              f"{stats['mean_grad_hard_neg']:10.6f} {stats['mean_grad_easy_neg']:10.6f} "
              f"{stats['ratio_hard_vs_easy']:8.1f}x {stats['mean_grad_pos']:10.6f}")

In [ ]:
# Gradient magnitude histograms: positive vs negative entries
fig, axes = plt.subplots(1, len(loss_fns), figsize=(5 * len(loss_fns), 4), sharey=True)
if len(loss_fns) == 1: axes = [axes]

for ax, (name, fn) in zip(axes, loss_fns.items()):
    grad_mat, _ = compute_sim_gradients(fn, _img_b, _txt_b, _pids_b)
    g_pos = grad_mat[_pos.bool()].abs().numpy()
    g_neg = grad_mat[_neg.bool()].abs().numpy()
    
    ax.hist(g_neg, bins=50, alpha=0.6, color='red', label='Negative', density=True)
    ax.hist(g_pos, bins=30, alpha=0.6, color='green', label='Positive', density=True)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('|grad|')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Density')
plt.suptitle('Gradient Magnitude: Positive vs Negative Pairs', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Section 4.5: NACIR — Noise-Aware Circle Loss Validation (Idea C)

Validates the **Unified Noise-Aware Circle Loss** before launching a full training run.

**Checks in order:**
1. **Degenerate equivalence** — NACIR with both detectors off must equal vanilla Circle Loss exactly (sanity check; diff should be 0).
2. **FN detection** — Fit Gaussians to the batch's `s_p`/`s_n` distributions, compute `P(FN|s)` for each negative, overlay the posterior curve on the similarity histogram.
3. **FP detection** — Compute per-sample Circle Loss on the full test set, fit a 2-component GMM, visualize clean/noisy separation.
4. **Gradient analysis** — Compare top-10% hard-negative gradient energy between vanilla Circle Loss and NACIR with FN detection active. **Good sign:** FN detection suppresses gradient on overlap-zone negatives (likely FN) while preserving it on genuine hard negatives.

**Greenlight criteria for full training:**
- Degenerate equivalence diff < 1e-5
- `P(FN|s)` curve shows clear rise in the overlap zone (not flat)
- GMM separation > 1.0 (if < 1.0 → fallback path will trigger; still safe but FP detection won't help)
- Gradient analysis shows reduced energy on top hard negatives without collapsing total gradient norm


In [ ]:
# --- NACIR setup: reuse the same batch as Section 3 ---
from model.objectives import compute_noise_aware_circle, _bayesian_fn_prob
from model.noise_aware import NoiseAwareCircleState

NACIR_M = 0.25
NACIR_GAMMA = 128
NACIR_EPS_N = 0.1
NACIR_EPS_P = 0.2
NACIR_FN_PRIOR = 0.01

print(f"NACIR hyperparameters: m={NACIR_M}, gamma={NACIR_GAMMA}, "
      f"eps_n={NACIR_EPS_N}, eps_p={NACIR_EPS_P}, fn_prior={NACIR_FN_PRIOR}")
print(f"Using same batch as Section 3: B={B}, unique PIDs={pids.unique().numel()}")


In [ ]:
# --- 1. Degenerate equivalence: NACIR with detectors OFF == vanilla Circle Loss ---
_img = img_f.detach().clone().requires_grad_(True)
_txt = txt_f.detach().clone().requires_grad_(True)

vanilla = compute_cross_modal_circle(_img, _txt, pids, m=NACIR_M, gamma=NACIR_GAMMA)
nacir_off, diag_off = compute_noise_aware_circle(
    _img, _txt, pids,
    m=NACIR_M, gamma=NACIR_GAMMA,
    fn_stats=None, clean_weights=None,
    epsilon_n=NACIR_EPS_N, epsilon_p=NACIR_EPS_P,
)

diff = abs(vanilla.item() - nacir_off.item())
print(f"Vanilla Circle Loss            : {vanilla.item():.6f}")
print(f"NACIR (both detectors off)     : {nacir_off.item():.6f}")
print(f"Abs diff                       : {diff:.2e}")
assert diff < 1e-4, f"NACIR must equal vanilla when detectors off (diff={diff})"
print(f"[PASS] Degenerate equivalence — safe rollback via NACIR: false")
print()
print(f"Diagnostic keys: {sorted(diag_off.keys())}")
print(f"  per_sample_loss shape   : {tuple(diag_off['per_sample_loss'].shape)}")
print(f"  s_p count               : {diag_off['s_p'].numel()}")
print(f"  s_n count               : {diag_off['s_n'].numel()}")


In [ ]:
# --- 2. FN detection: fit Gaussians and visualize P(FN|s) ---
# Build fn_stats from the batch's own similarity distributions
with torch.no_grad():
    sim_mat = img_f @ txt_f.t()
    pid_eq = torch.eq(pids.view(-1, 1), pids.view(1, -1))
    s_p_batch = sim_mat[pid_eq]
    s_n_batch = sim_mat[~pid_eq]

fn_stats = {
    'mu_pos':    s_p_batch.mean().item(),
    'sigma_pos': s_p_batch.std().item(),
    'mu_neg':    s_n_batch.mean().item(),
    'sigma_neg': s_n_batch.std().item(),
    'fn_prior':  NACIR_FN_PRIOR,
}
print(f"Fitted Gaussians:")
print(f"  Positives  : mu={fn_stats['mu_pos']:.4f}, sigma={fn_stats['sigma_pos']:.4f}  ({s_p_batch.numel()} pairs)")
print(f"  Negatives  : mu={fn_stats['mu_neg']:.4f}, sigma={fn_stats['sigma_neg']:.4f}  ({s_n_batch.numel()} pairs)")
print(f"  Separation : {(fn_stats['mu_pos'] - fn_stats['mu_neg']) / (fn_stats['sigma_pos'] + fn_stats['sigma_neg']):.3f}")

# Compute NACIR with FN detection active
nacir_fn, diag_fn = compute_noise_aware_circle(
    _img, _txt, pids,
    m=NACIR_M, gamma=NACIR_GAMMA,
    fn_stats=fn_stats, clean_weights=None,
    epsilon_n=NACIR_EPS_N, epsilon_p=NACIR_EPS_P,
)
print(f"\nNACIR with FN detection        : {nacir_fn.item():.6f}")
print(f"  mean P(FN|s_n)               : {diag_fn['fn_prob_mean']:.4f}")
print(f"  mean alpha_n suppression     : {diag_fn['alpha_n_scale_mean']:.4f}  (1.0 = no suppression)")
print(f"  delta vs vanilla             : {nacir_fn.item() - vanilla.item():+.4f}")

# --- Visualize P(FN|s) curve overlaid on similarity histograms ---
with torch.no_grad():
    s_grid = torch.linspace(
        min(s_n_batch.min().item(), s_p_batch.min().item()) - 0.05,
        max(s_n_batch.max().item(), s_p_batch.max().item()) + 0.05,
        400,
    )
    p_fn_curve = _bayesian_fn_prob(
        s_grid,
        mu_pos=fn_stats['mu_pos'], sigma_pos=fn_stats['sigma_pos'],
        mu_neg=fn_stats['mu_neg'], sigma_neg=fn_stats['sigma_neg'],
        fn_prior=fn_stats['fn_prior'],
    )

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.hist(s_n_batch.numpy(), bins=60, alpha=0.5, color='red',   density=True, label='Negatives')
ax1.hist(s_p_batch.numpy(), bins=30, alpha=0.5, color='green', density=True, label='Positives')
ax1.set_xlabel('Similarity  s')
ax1.set_ylabel('Density', color='black')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(s_grid.numpy(), p_fn_curve.numpy(), 'b-', linewidth=2.5, label='P(FN|s)')
ax2.set_ylabel('P(FN | s)', color='blue')
ax2.set_ylim(-0.02, 1.02)
ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.7)
ax2.legend(loc='upper right')

plt.title(f'FN Detection: Bayesian Posterior over Similarity Histograms  (B={B})')
plt.tight_layout()
plt.show()


In [ ]:
# --- 3. FP detection: GMM on per-sample Circle Loss over the full test set ---
# Compute per-sample loss using the diagnostics output (row-wise Circle Loss decomposition)
from model.noise_aware import NoiseAwareCircleState

# Build a state with the full test set size to hold all per-sample losses
FULL_N = W['image_feats'].shape[0]

class _Cfg:
    """Duck-typed config stub for NoiseAwareCircleState."""
    def get(self, k, d): return d

nstate = NoiseAwareCircleState(num_train_samples=FULL_N, config=_Cfg())

# Iterate mini-batches over the full test set, feeding per_sample_loss into the state
BATCH_FOR_TRACK = 128
all_feats_i = W['image_feats']
all_feats_t = W['text_feats']
all_pids    = W['image_pids']

with torch.no_grad():
    for start in range(0, FULL_N, BATCH_FOR_TRACK):
        end = min(start + BATCH_FOR_TRACK, FULL_N)
        idx = torch.arange(start, end)
        if idx.numel() < 4:
            continue  # too small for meaningful per-row Circle
        b_img = all_feats_i[idx]
        b_txt = all_feats_t[idx]
        b_pid = all_pids[idx]
        # Skip batches with no positive pairs
        pid_eq = torch.eq(b_pid.view(-1, 1), b_pid.view(1, -1))
        if not pid_eq.any() or (pid_eq.sum() - len(b_pid)) < 1:
            continue
        _, d = compute_noise_aware_circle(
            b_img, b_txt, b_pid,
            m=NACIR_M, gamma=NACIR_GAMMA,
            fn_stats=None, clean_weights=None,
        )
        nstate.update_sample_losses(idx, d['per_sample_loss'])

n_seen = int(nstate.sample_seen.sum().item())
print(f"Per-sample Circle Loss tracked for {n_seen}/{FULL_N} samples")

# --- Fit GMM ---
gmm_diag = nstate.refit_gmm()
print(f"\nGMM fit results:")
for k, v in gmm_diag.items():
    print(f"  {k:20s} : {v}")

# --- Visualize ---
losses_arr = nstate.ema_loss[nstate.sample_seen].numpy()
clean_weights_arr = nstate.clean_weights[nstate.sample_seen].numpy()

# Pure-numpy Gaussian PDF (no scipy dependency)
def _gauss_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: loss distribution + GMM components
axes[0].hist(losses_arr, bins=80, alpha=0.6, color='steelblue', density=True,
             label=f'Per-sample loss (n={n_seen})')
if gmm_diag['fallback'] < 0.5:
    x_grid = np.linspace(losses_arr.min(), losses_arr.max(), 300)
    axes[0].plot(x_grid,
                 gmm_diag['pi_clean'] * _gauss_pdf(x_grid, gmm_diag['mu_clean'], gmm_diag['sigma_clean']),
                 'g-', lw=2, label=f"Clean  N({gmm_diag['mu_clean']:.2f}, {gmm_diag['sigma_clean']:.3f})")
    axes[0].plot(x_grid,
                 gmm_diag['pi_noisy'] * _gauss_pdf(x_grid, gmm_diag['mu_noisy'], gmm_diag['sigma_noisy']),
                 'r-', lw=2, label=f"Noisy  N({gmm_diag['mu_noisy']:.2f}, {gmm_diag['sigma_noisy']:.3f})")
axes[0].set_xlabel('Per-sample Circle Loss')
axes[0].set_ylabel('Density')
axes[0].set_title(f"GMM fit — separation={gmm_diag['separation']:.3f}"
                  + (" (FALLBACK)" if gmm_diag['fallback'] > 0.5 else ""))
axes[0].legend()

# Right: clean_weights distribution
axes[1].hist(clean_weights_arr, bins=50, color='purple', alpha=0.7)
axes[1].axvline(NACIR_EPS_P, color='red', linestyle='--', label=f'eps_p = {NACIR_EPS_P} (floor)')
axes[1].set_xlabel('Clean probability  w_i')
axes[1].set_ylabel('Number of samples')
axes[1].set_title(f'Clean-weight distribution  (mean={clean_weights_arr.mean():.3f})')
axes[1].legend()
plt.tight_layout()
plt.show()

if gmm_diag['fallback'] > 0.5:
    print("\n[WARNING] GMM fallback active — separation below threshold.")
    print("This is EXPECTED on a well-trained checkpoint (the model HAS memorized)")
    print("if there is little actual label noise. For a LoRA checkpoint, it may also")
    print("indicate that memorization is too weak for GMM to detect FP reliably.")
    print("Training will still proceed safely (clean_weights = 1.0 -> FP detection inactive).")


In [ ]:
# --- 4. Gradient Analysis: NACIR vs vanilla Circle Loss ---
# Wrap NACIR variants for the sim-matrix gradient analyzer used in Section 4.

# NACIR with NO detection (should equal _cross_circle_from_sim exactly)
def _nacir_off_from_sim(sim, pos_mask, neg_mask, pids, **kw):
    s_p = sim[pos_mask.bool()]
    s_n = sim[neg_mask.bool()]
    if s_p.numel() == 0 or s_n.numel() == 0:
        return torch.tensor(0.0, requires_grad=True)
    m, gamma = NACIR_M, NACIR_GAMMA
    alpha_p = torch.clamp_min(-s_p.detach() + 1 + m, min=0.)
    alpha_n = torch.clamp_min(s_n.detach() + m, min=0.)
    logit_p = -gamma * alpha_p * (s_p - (1 - m))
    logit_n =  gamma * alpha_n * (s_n - m)
    return F.softplus(torch.logsumexp(logit_p, dim=0) + torch.logsumexp(logit_n, dim=0))

# NACIR with FN detection ACTIVE
def _nacir_fn_from_sim(sim, pos_mask, neg_mask, pids, **kw):
    s_p = sim[pos_mask.bool()]
    s_n = sim[neg_mask.bool()]
    if s_p.numel() == 0 or s_n.numel() == 0:
        return torch.tensor(0.0, requires_grad=True)
    m, gamma = NACIR_M, NACIR_GAMMA
    alpha_p = torch.clamp_min(-s_p.detach() + 1 + m, min=0.)
    alpha_n = torch.clamp_min(s_n.detach() + m, min=0.)
    # Bayesian FN suppression (using fn_stats from the previous cell)
    fn_probs = _bayesian_fn_prob(
        s_n.detach(),
        mu_pos=fn_stats['mu_pos'], sigma_pos=fn_stats['sigma_pos'],
        mu_neg=fn_stats['mu_neg'], sigma_neg=fn_stats['sigma_neg'],
        fn_prior=fn_stats['fn_prior'],
    )
    alpha_n = alpha_n * torch.clamp_min(1.0 - fn_probs, min=NACIR_EPS_N)
    logit_p = -gamma * alpha_p * (s_p - (1 - m))
    logit_n =  gamma * alpha_n * (s_n - m)
    return F.softplus(torch.logsumexp(logit_p, dim=0) + torch.logsumexp(logit_n, dim=0))

# Extend the loss_fns dict from Section 4 with NACIR variants
loss_fns_nacir = {
    'Cross-Modal Circle': lambda *a, **k: _cross_circle_from_sim(*a, m=NACIR_M, gamma=NACIR_GAMMA),
    'NACIR (off) [sanity]':  _nacir_off_from_sim,
    'NACIR + FN detection':  _nacir_fn_from_sim,
}

print(f"{'Loss':<25} {'Energy% Hard':>14} {'Mean Hard':>10} {'Mean Easy':>10} {'Ratio':>8} {'Mean Pos':>10}")
print("-" * 80)
row_stats = {}
for name, fn in loss_fns_nacir.items():
    grad_mat, loss_val = compute_sim_gradients(fn, _img_b, _txt_b, _pids_b)
    stats = analyze_hard_neg_gradients(grad_mat, _pos, _neg)
    row_stats[name] = stats
    if stats:
        print(f"{name:<25} {stats['hard_neg_energy_pct']:13.1f}% "
              f"{stats['mean_grad_hard_neg']:10.6f} {stats['mean_grad_easy_neg']:10.6f} "
              f"{stats['ratio_hard_vs_easy']:8.1f}x {stats['mean_grad_pos']:10.6f}")

# --- Diagnostic: compare vanilla vs NACIR+FN ---
vanilla_name = 'Cross-Modal Circle'
nacir_name   = 'NACIR + FN detection'
if vanilla_name in row_stats and nacir_name in row_stats:
    v = row_stats[vanilla_name]
    n = row_stats[nacir_name]
    delta_hard_pct = n['hard_neg_energy_pct'] - v['hard_neg_energy_pct']
    delta_ratio    = n['ratio_hard_vs_easy']  - v['ratio_hard_vs_easy']
    print()
    print(f"Delta NACIR+FN vs Vanilla:")
    print(f"  hard_neg_energy_pct : {delta_hard_pct:+.2f} pp")
    print(f"  ratio_hard_vs_easy  : {delta_ratio:+.2f}x")
    if delta_hard_pct < 0:
        print("  → FN detection reduces gradient energy on top hard negatives (softening suspected FNs).")
    else:
        print("  → FN detection did NOT reduce hard-neg energy — check fn_prior / overlap is real.")


In [ ]:
# --- 5. Gradient magnitude histograms: vanilla vs NACIR+FN side-by-side ---
fig, axes = plt.subplots(1, len(loss_fns_nacir), figsize=(5 * len(loss_fns_nacir), 4), sharey=True)
if len(loss_fns_nacir) == 1: axes = [axes]

for ax, (name, fn) in zip(axes, loss_fns_nacir.items()):
    grad_mat, _ = compute_sim_gradients(fn, _img_b, _txt_b, _pids_b)
    g_pos = grad_mat[_pos.bool()].abs().numpy()
    g_neg = grad_mat[_neg.bool()].abs().numpy()
    ax.hist(g_neg, bins=50, alpha=0.6, color='red',   label='Negative', density=True)
    ax.hist(g_pos, bins=30, alpha=0.6, color='green', label='Positive', density=True)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('|grad|')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Density')
plt.suptitle('NACIR Gradient Magnitudes: Positive vs Negative Pairs', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# --- 6. Greenlight summary ---
print("=" * 70)
print("  NACIR (Idea C) Validation Summary")
print("=" * 70)

checks = []

# 1. Degenerate equivalence
degen_ok = diff < 1e-4
checks.append(("Degenerate equivalence (NACIR off == Circle)", degen_ok, f"diff={diff:.2e}"))

# 2. FN curve is non-trivial
fn_curve_nontrivial = (p_fn_curve.max() - p_fn_curve.min()).item() > 0.1
checks.append(("FN posterior P(FN|s) is non-flat",           fn_curve_nontrivial,
               f"range=[{p_fn_curve.min().item():.3f}, {p_fn_curve.max().item():.3f}]"))

# 3. GMM succeeds OR gracefully falls back
gmm_ok_or_safe_fallback = (gmm_diag['fallback'] > 0.5) or (gmm_diag['separation'] > 1.0)
checks.append(("GMM succeeds or safely falls back",          gmm_ok_or_safe_fallback,
               f"sep={gmm_diag['separation']:.3f}, fallback={bool(gmm_diag['fallback'])}"))

# 4. FN detection does not collapse total gradient
nacir_fn_stats = row_stats.get('NACIR + FN detection', {})
vanilla_stats  = row_stats.get('Cross-Modal Circle', {})
total_grad_nacir   = nacir_fn_stats.get('mean_grad_hard_neg', 0) + nacir_fn_stats.get('mean_grad_easy_neg', 0)
total_grad_vanilla = vanilla_stats.get('mean_grad_hard_neg', 0)  + vanilla_stats.get('mean_grad_easy_neg', 0)
no_collapse = (total_grad_vanilla == 0) or (total_grad_nacir / total_grad_vanilla > 0.3)
checks.append(("No gradient collapse (NACIR / vanilla > 0.3)", no_collapse,
               f"ratio={total_grad_nacir / (total_grad_vanilla + 1e-12):.3f}"))

for name, ok, detail in checks:
    mark = "[PASS]" if ok else "[FAIL]"
    print(f"  {mark}  {name:<48}  {detail}")

all_ok = all(c[1] for c in checks)
print()
if all_ok:
    print("  GREENLIGHT: all checks passed. Safe to launch a full NACIR training run.")
    print()
    print("  To enable:  set  NACIR: true  in  config/loss/cir_msiglip.yaml")
else:
    print("  STOP: at least one check failed. Investigate before training.")


---
## Section 5: Embedding Space Visualization
Visualize the learned embedding geometry: t-SNE clustering, intra/inter-class distributions, and the (s_n, s_p) diagnostic scatter.

**Good signs:** Tight t-SNE clusters with mixed modalities. Most points above the y=x line in the scatter plot.

In [ ]:
# t-SNE visualization (subset of identities)
from sklearn.manifold import TSNE

N_PIDS_VIS = 15  # Number of identities to visualize
unique_pids = W['image_pids'].unique()
sel_pids = unique_pids[torch.randperm(len(unique_pids))[:N_PIDS_VIS]]

# Select image and text embeddings for these pids
img_mask = torch.isin(W['image_pids'], sel_pids)
txt_mask = torch.isin(W['text_pids'], sel_pids)

img_sel = W['image_feats'][img_mask]
txt_sel = W['text_feats'][txt_mask]
img_pids_sel = W['image_pids'][img_mask]
txt_pids_sel = W['text_pids'][txt_mask]

# Concatenate for t-SNE
all_feats = torch.cat([img_sel, txt_sel]).numpy()
all_pids  = torch.cat([img_pids_sel, txt_pids_sel]).numpy()
modality  = ['img'] * len(img_sel) + ['txt'] * len(txt_sel)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
proj = tsne.fit_transform(all_feats)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.get_cmap('tab20', N_PIDS_VIS)
pid_to_idx = {pid: i for i, pid in enumerate(sel_pids.numpy())}

for i, (x, y) in enumerate(proj):
    pid_idx = pid_to_idx[all_pids[i]]
    marker = 'o' if modality[i] == 'img' else '^'
    ax.scatter(x, y, c=[cmap(pid_idx)], marker=marker, s=40, alpha=0.7, edgecolors='none')

# Legend
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], marker='o', color='gray', linestyle='', markersize=8, label='Image'),
           Line2D([0],[0], marker='^', color='gray', linestyle='', markersize=8, label='Text')]
ax.legend(handles=handles, loc='upper right')
ax.set_title(f't-SNE of {N_PIDS_VIS} Identities (Image + Text)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

In [ ]:
# Intra-class vs Inter-class similarity distributions (cross-modal)
sim_full = W['text_feats'] @ W['image_feats'].t()
pos_full = torch.eq(W['text_pids'].view(-1, 1), W['image_pids'].view(1, -1))

intra = sim_full[pos_full].numpy()
# Sample inter-class to avoid memory issues
inter_idx = torch.where(~pos_full)
sample_n = min(len(intra) * 10, len(inter_idx[0]))
sample_i = torch.randperm(len(inter_idx[0]))[:sample_n]
inter = sim_full[inter_idx[0][sample_i], inter_idx[1][sample_i]].numpy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(inter, bins=100, alpha=0.6, color='red', label=f'Inter-class (n={len(inter):,})', density=True)
ax.hist(intra, bins=50, alpha=0.6, color='green', label=f'Intra-class (n={len(intra):,})', density=True)
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Density')
ax.set_title('Cross-Modal: Intra-class vs Inter-class Similarity')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Diagnostic scatter plot: (s_n_max, s_p_max) per query
# Replicates the analysis from visualize_distribution.py
sp_list, sn_list = [], []
for i in range(sim_full.shape[0]):
    q_pid = W['text_pids'][i]
    q_sims = sim_full[i]
    is_pos = (W['image_pids'] == q_pid)
    is_neg = ~is_pos
    if is_pos.sum() > 0 and is_neg.sum() > 0:
        sp_list.append(q_sims[is_pos].max().item())
        sn_list.append(q_sims[is_neg].max().item())
sp_arr = np.array(sp_list)
sn_arr = np.array(sn_list)

fig, ax = plt.subplots(figsize=(8, 8))
is_correct = sp_arr > sn_arr
ax.scatter(sn_arr[is_correct], sp_arr[is_correct], s=15, c='green', alpha=0.15, label='Correct', edgecolors='none')
ax.scatter(sn_arr[~is_correct], sp_arr[~is_correct], s=15, c='red', alpha=0.15, label='Incorrect', edgecolors='none')
ax.scatter(np.mean(sn_arr), np.mean(sp_arr), c='black', marker='X', s=200, edgecolors='white', linewidth=2, zorder=10, label='Centroid')

limit = 0.8
ax.plot([0, limit], [0, limit], 'k--', linewidth=2, label='y=x')

# Circle Loss margin arc
target_margin = 0.25
if target_margin > 0:
    r = (1 - target_margin) / np.sqrt(2)
    xc = np.linspace(0, r, 200)
    yc = 1 - np.sqrt(np.clip(r**2 - xc**2, 0, None))
    valid = (xc <= limit) & (yc >= 0)
    ax.plot(xc[valid], yc[valid], color='orange', linestyle='--', linewidth=2.5, label=f'Target (m={target_margin})')

pass_rate = np.sum(sp_arr > sn_arr) / len(sp_arr) * 100
avg_gap = np.mean(sp_arr - sn_arr)

ax.set_xlim(0, limit); ax.set_ylim(0, limit)
ax.set_xlabel(r'$s_n$ (Hardest Negative)', fontweight='bold')
ax.set_ylabel(r'$s_p$ (Best Positive)', fontweight='bold')
ax.set_title('Diagnostic: Positive vs Hardest Negative per Query')
ax.text(0.04, 0.96, f"Rank-1: {pass_rate:.1f}%\nAvg Gap: {avg_gap:.3f}",
        transform=ax.transAxes, verticalalignment='top', fontsize=13,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9, edgecolor='gray'))
ax.legend(loc='lower right')
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

---
## Section 6: Quick Retrieval Metrics
Full metrics on the loaded embeddings + "what-if" transforms + error analysis.

**Good signs:** Metrics match your training logs. Error cases reveal systematic patterns you can target.

In [ ]:
# Full retrieval metrics (T2I and I2T)
sim_t2i = W['text_feats'] @ W['image_feats'].t()
sim_i2t = sim_t2i.t()

t2i_cmc, t2i_mAP, t2i_mINP, t2i_indices = rank(sim_t2i, W['text_pids'], W['image_pids'], max_rank=10, get_mAP=True)
i2t_cmc, i2t_mAP, i2t_mINP, _ = rank(sim_i2t, W['image_pids'], W['text_pids'], max_rank=10, get_mAP=True)

print(f"{'Task':<6} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'mAP':>8} {'mINP':>8}")
print("-" * 48)
print(f"{'T2I':<6} {t2i_cmc[0]:8.2f} {t2i_cmc[4]:8.2f} {t2i_cmc[9]:8.2f} {t2i_mAP:8.2f} {t2i_mINP:8.2f}")
print(f"{'I2T':<6} {i2t_cmc[0]:8.2f} {i2t_cmc[4]:8.2f} {i2t_cmc[9]:8.2f} {i2t_mAP:8.2f} {i2t_mINP:8.2f}")

In [ ]:
# "What-if" metrics: apply a transform and re-evaluate
def what_if_metrics(image_feats, text_feats, image_pids, text_pids, transform_fn=None, label="Default"):
    """
    Apply an optional transform to embeddings, then compute metrics.
    transform_fn: callable(img_feats, txt_feats) -> (img_feats, txt_feats)
    """
    img_f = image_feats.clone()
    txt_f = text_feats.clone()
    if transform_fn is not None:
        img_f, txt_f = transform_fn(img_f, txt_f)
    img_f = F.normalize(img_f, p=2, dim=1)
    txt_f = F.normalize(txt_f, p=2, dim=1)
    sim = txt_f @ img_f.t()
    cmc, mAP, mINP, _ = rank(sim, text_pids, image_pids, max_rank=10, get_mAP=True)
    print(f"{label:<30} R@1={cmc[0]:.2f}  R@5={cmc[4]:.2f}  mAP={mAP:.2f}")
    return cmc, mAP, mINP

# Example: test different logit scales (temperature scaling on raw features)
what_if_metrics(W['image_feats'], W['text_feats'], W['image_pids'], W['text_pids'], label="Baseline (no transform)")

# Example: add a small random projection
# def random_proj(img, txt):
#     proj = torch.randn(768, 768) * 0.01 + torch.eye(768)
#     return img @ proj, txt @ proj
# what_if_metrics(W['image_feats'], W['text_feats'], W['image_pids'], W['text_pids'], transform_fn=random_proj, label="Random projection")

In [ ]:
# Error analysis: top failure cases (T2I)
pred_labels = W['image_pids'][t2i_indices]
matches = pred_labels.eq(W['text_pids'].view(-1, 1))
failures = torch.where(~matches[:, 0])[0]  # Queries where R@1 fails

N_FAILURES = 20
print(f"Total R@1 failures: {len(failures)} / {len(W['text_pids'])}")
print(f"\nTop {N_FAILURES} failure cases:")
print(f"{'Query#':<8} {'True PID':<10} {'Pred PID@1':<12} {'Sim(true)':<10} {'Sim(pred)':<10} {'Gap':<8}")
print("-" * 60)

for i, q_idx in enumerate(failures[:N_FAILURES]):
    q_idx = q_idx.item()
    true_pid = W['text_pids'][q_idx].item()
    pred_pid = W['image_pids'][t2i_indices[q_idx, 0]].item()
    # Best positive similarity
    pos_mask_q = (W['image_pids'] == true_pid)
    sim_true = sim_t2i[q_idx][pos_mask_q].max().item()
    sim_pred = sim_t2i[q_idx, t2i_indices[q_idx, 0]].item()
    print(f"{q_idx:<8} {true_pid:<10} {pred_pid:<12} {sim_true:<10.4f} {sim_pred:<10.4f} {sim_true - sim_pred:<8.4f}")

---
## Section 7: Checkpoint A/B Comparison
Load two checkpoints and compare everything side-by-side.  
Set `CKPT_PATH_B` in Section 0 before running this section.

In [ ]:
assert CKPT_PATH_B is not None, "Set CKPT_PATH_B in Section 0 before running this section."

# Load second checkpoint (reuses same config/data)
_, dm_b, model_b = load_model_and_data(CONFIG_PATH, CKPT_PATH_B)
W_B = extract_embeddings(model_b, dm_b, config)

# Clean up model B to save memory
del model_b
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Side-by-side metrics comparison
def _compute_all_metrics(w, label):
    sim = w['text_feats'] @ w['image_feats'].t()
    cmc, mAP, mINP, indices = rank(sim, w['text_pids'], w['image_pids'], max_rank=10, get_mAP=True)
    return {'label': label, 'R@1': cmc[0].item(), 'R@5': cmc[4].item(), 'R@10': cmc[9].item(),
            'mAP': mAP.item(), 'mINP': mINP.item(), 'indices': indices}

m_a = _compute_all_metrics(W, "Checkpoint A")
m_b = _compute_all_metrics(W_B, "Checkpoint B")

print(f"{'Metric':<10} {'Ckpt A':>10} {'Ckpt B':>10} {'Delta':>10}")
print("-" * 42)
for key in ['R@1', 'R@5', 'R@10', 'mAP', 'mINP']:
    delta = m_b[key] - m_a[key]
    sign = "+" if delta >= 0 else ""
    print(f"{key:<10} {m_a[key]:10.2f} {m_b[key]:10.2f} {sign}{delta:9.2f}")

In [ ]:
# Overlaid similarity distributions from both checkpoints
sim_a = W['text_feats'] @ W['image_feats'].t()
sim_b = W_B['text_feats'] @ W_B['image_feats'].t()
pos_a = torch.eq(W['text_pids'].view(-1,1), W['image_pids'].view(1,-1))
pos_b = torch.eq(W_B['text_pids'].view(-1,1), W_B['image_pids'].view(1,-1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sim_m, pos_m, title in [(axes[0], sim_a, pos_a, "Checkpoint A"), (axes[1], sim_b, pos_b, "Checkpoint B")]:
    sp = sim_m[pos_m].numpy()
    sn_sample = sim_m[~pos_m].flatten()
    sn_sample = sn_sample[torch.randperm(len(sn_sample))[:len(sp)*10]].numpy()
    ax.hist(sn_sample, bins=80, alpha=0.5, color='red', label='Negative', density=True)
    ax.hist(sp, bins=40, alpha=0.5, color='green', label='Positive', density=True)
    ax.set_title(title)
    ax.set_xlabel('Cosine Similarity')
    ax.legend()
plt.suptitle('Similarity Distribution Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Per-query rank improvement: scatter plot
rank_a = torch.zeros(sim_a.shape[0], dtype=torch.long)
rank_b = torch.zeros(sim_b.shape[0], dtype=torch.long)

indices_a = torch.argsort(sim_a, dim=1, descending=True)
indices_b = torch.argsort(sim_b, dim=1, descending=True)

for i in range(sim_a.shape[0]):
    true_pid = W['text_pids'][i]
    # Find rank in A
    pred_a = W['image_pids'][indices_a[i]]
    rank_a[i] = (pred_a == true_pid).nonzero(as_tuple=True)[0][0] + 1 if (pred_a == true_pid).any() else sim_a.shape[1]
    # Find rank in B
    pred_b = W_B['image_pids'][indices_b[i]]
    rank_b[i] = (pred_b == true_pid).nonzero(as_tuple=True)[0][0] + 1 if (pred_b == true_pid).any() else sim_b.shape[1]

improved = (rank_b < rank_a).sum().item()
regressed = (rank_b > rank_a).sum().item()
unchanged = (rank_b == rank_a).sum().item()

fig, ax = plt.subplots(figsize=(7, 7))
max_r = min(50, rank_a.max().item(), rank_b.max().item())
ax.scatter(rank_a.clamp(max=max_r).numpy(), rank_b.clamp(max=max_r).numpy(), s=10, alpha=0.3)
ax.plot([0, max_r], [0, max_r], 'k--', linewidth=1.5, label='No change')
ax.set_xlabel('Rank (Checkpoint A)')
ax.set_ylabel('Rank (Checkpoint B)')
ax.set_title('Per-Query Rank Comparison')
ax.text(0.05, 0.95, f"Improved: {improved}\nRegressed: {regressed}\nUnchanged: {unchanged}",
        transform=ax.transAxes, va='top', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 8: Mini Fine-Tune Sanity Check (Optional)
Run 1-3 epochs on a tiny subset with your new loss to verify it produces usable gradients.  
**Warning:** This modifies the model in-memory. Re-run Section 1 afterward to reset.

**Good signs:** Loss decreases. R@1 improves or stays stable. No NaN/collapse.

In [ ]:
# Pre-fine-tune R@1 baseline
pre_sim = W['text_feats'] @ W['image_feats'].t()
pre_cmc, _, _, _ = rank(pre_sim, W['text_pids'], W['image_pids'], max_rank=10, get_mAP=True)
print(f"Pre fine-tune R@1: {pre_cmc[0]:.2f}")

In [ ]:
# Mini fine-tune loop
from data.bases import ImageTextDataset
from data.augmentation.transform import build_image_aug_pool, build_text_aug_pool

MINI_EPOCHS = 2
MINI_BATCH = 16
MINI_LR = 1e-5
MINI_BATCHES_PER_EPOCH = 10

# Build a small train loader
train_set = ImageTextDataset(
    dataset=dm.dataset.train[:MINI_BATCH * MINI_BATCHES_PER_EPOCH],
    tokenizer=dm.tokenizer,
    image_augmentation_pool=build_image_aug_pool(config.aug.img.augment_cfg),
    text_augmentation_pool=build_text_aug_pool(config.aug.text.augment_cfg),
    image_random_k=config.aug.image_random_k,
    text_random_k=config.aug.text_random_k,
    image_size=config.aug.img.size,
    is_train=True,
    mean=config.aug.img.mean,
    std=config.aug.img.std,
)
train_loader = DataLoader(train_set, batch_size=MINI_BATCH, shuffle=True, num_workers=2)

model.train()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=MINI_LR)

loss_history = []
for epoch in range(MINI_EPOCHS):
    epoch_losses = []
    for i, batch in enumerate(train_loader):
        if i >= MINI_BATCHES_PER_EPOCH:
            break
        imgs = batch['images'].to(DEVICE)
        input_ids = batch['caption_input_ids'].to(DEVICE)
        attn_mask = batch['caption_attention_mask'].to(DEVICE)
        batch_pids = batch['pids'].to(DEVICE)

        img_feats = model.get_image_features(imgs)
        txt_feats = model.get_text_features({'input_ids': input_ids, 'attention_mask': attn_mask})
        img_feats = F.normalize(img_feats, p=2, dim=1)
        txt_feats = F.normalize(txt_feats, p=2, dim=1)

        loss = my_new_loss(img_feats, txt_feats, batch_pids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())
    avg_loss = np.mean(epoch_losses)
    loss_history.extend(epoch_losses)
    print(f"Epoch {epoch+1}/{MINI_EPOCHS} | Avg Loss: {avg_loss:.4f}")

model.eval()

# Plot loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(loss_history, marker='o', markersize=3)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Mini Fine-Tune Loss Curve')
plt.tight_layout()
plt.show()

In [ ]:
# Post fine-tune: re-extract and compare R@1
W_post = extract_embeddings(model, dm, config)
post_sim = W_post['text_feats'] @ W_post['image_feats'].t()
post_cmc, post_mAP, post_mINP, _ = rank(post_sim, W_post['text_pids'], W_post['image_pids'], max_rank=10, get_mAP=True)

print(f"{'':>15} {'R@1':>8} {'R@5':>8} {'mAP':>8}")
print(f"{'Pre fine-tune':>15} {pre_cmc[0]:8.2f} {pre_cmc[4]:8.2f} {'--':>8}")
print(f"{'Post fine-tune':>15} {post_cmc[0]:8.2f} {post_cmc[4]:8.2f} {post_mAP:8.2f}")
delta = post_cmc[0] - pre_cmc[0]
print(f"\nR@1 delta: {'+' if delta >= 0 else ''}{delta:.2f}")
if delta < -2:
    print("WARNING: R@1 dropped significantly. The loss may be causing collapse.")

---
## Quick Reference: "Good Signs" Cheatsheet

| Signal | Where | Good | Bad |
|--------|-------|------|-----|
| Gradient energy on hard negs | Section 4 table | Higher ratio than N-ITC | Lower or zero |
| Pos/neg separation | Section 2 histogram | Clear gap, low overlap | Overlapping distributions |
| Loss value | Section 3 table | Finite, similar scale to existing | NaN, inf, or 0 |
| R@1 after mini fine-tune | Section 8 | Improves or stable | Collapses (>2pt drop) |
| t-SNE clusters | Section 5 | Tight clusters, mixed modality | Scattered, no structure |
| (s_p, s_n) scatter | Section 5 | Most points above y=x | Many below y=x |

### Workflow
1. Write your new loss in **Section 3** template cell
2. Check gradient focus in **Section 4** — this is the key diagnostic
3. If gradients look good, run **Section 8** mini fine-tune
4. If R@1 holds or improves, commit to full training